In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    classification_report
)

In [3]:
df = pd.read_csv("../../data/processed/features.csv")

product_features = [
    "starting_capital",
    "account_age_months",
    "instrument",
    "risk_per_trade_pct",
    "trades_per_month",
    "uses_stop_loss",
    "avg_win_hold_days",
    "avg_loss_hold_days",
    "diversification",
    "follows_plan",
    "position_sizing_discipline"
]

product_df = df[product_features + ["blew_up"]].copy()

X = product_df.drop(columns=["blew_up"])
y = product_df["blew_up"]

print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts(normalize=True))

X shape: (50000, 11)
Target distribution:
blew_up
0    0.54926
1    0.45074
Name: proportion, dtype: float64


In [4]:
categorical_columns = ["instrument"]

numeric_columns = [
    column for column in X.columns
    if column not in categorical_columns
]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

preprocess = ColumnTransformer([
    ("numeric", MinMaxScaler(), numeric_columns),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [6]:
models = {
    "Logistic Regression": (
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000
        ),
        {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["lbfgs", "liblinear"]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "model__n_neighbors": [5, 10, 15, 25],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),

    "SVM": (
        SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True
        ),
        {
            "model__C": [0.1, 1, 10],
            "model__gamma": ["scale", 0.01, 0.1]
        }
    )
}

In [7]:
results = []
best_models = {}

for name, (model, parameters) in models.items():

    print(f"{name} çalışıyor...")

    pipeline = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameters,
        cv=5,
        scoring="balanced_accuracy",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    predictions = best_model.predict(X_test)
    probabilities = best_model.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "cv_balanced_accuracy": grid_search.best_score_,
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_balanced_accuracy": balanced_accuracy_score(
            y_test,
            predictions
        ),
        "test_roc_auc": roc_auc_score(y_test, probabilities),
        "best_parameters": grid_search.best_params_
    })

    best_models[name] = best_model

Logistic Regression çalışıyor...
KNN çalışıyor...
SVM çalışıyor...


/opt/anaconda3/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

In [8]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "test_roc_auc",
    ascending=False
)

,model,cv_balanced_accuracy,test_accuracy,test_balanced_accuracy,test_roc_auc,best_parameters
0,Logistic Regression,0.611570,0.5990,0.597712,0.638071,"{'model__C': 1, 'model__solver': 'liblinear'}"
2,SVM,0.610020,0.5988,0.596235,0.638068,"{'model__C': 10, 'model__gamma': 0.01}"
1,KNN,0.582332,0.5857,0.571706,0.605911,"{'model__n_neighbors': 25, 'model__p': 2, 'mod..."


In [12]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_recall_curve,
    classification_report,
    confusion_matrix
)

best_product_model = best_models["Logistic Regression"]

oof_probabilities = cross_val_predict(
    best_product_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

precision, recall, thresholds = precision_recall_curve(
    y_train,
    oof_probabilities
)

target_recall = 0.80

valid_thresholds = thresholds[recall[:-1] >= target_recall]

if len(valid_thresholds) > 0:
    product_threshold = float(valid_thresholds.max())
else:
    product_threshold = 0.50

print("Seçilen threshold:", product_threshold)

Seçilen threshold: 0.41662803044772917


In [11]:
test_probabilities = best_product_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_probabilities >= product_threshold
).astype(int)

print(classification_report(
    y_test,
    test_predictions,
    digits=3
))

print(confusion_matrix(
    y_test,
    test_predictions
))

              precision    recall  f1-score   support

           0      0.677     0.370     0.479      5493
           1      0.506     0.785     0.615      4507

    accuracy                          0.557     10000
   macro avg      0.592     0.578     0.547     10000
weighted avg      0.600     0.557     0.540     10000

[[2033 3460]
 [ 968 3539]]


In [13]:
import json
import joblib
from pathlib import Path

artifact_dir = Path("../../artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

# Seçilen model, tüm veriyle yeniden eğitiliyor
best_product_model.fit(X, y)

product_artifact_path = (
    artifact_dir / "product_blowup_risk_logistic_v1.joblib"
)

joblib.dump(best_product_model, product_artifact_path)

print("Model kaydedildi:")
print(product_artifact_path.resolve())

Model kaydedildi:
/Users/ccakir/Desktop/TradeMiror/artifacts/product_blowup_risk_logistic_v1.joblib


In [14]:
product_metrics = {
    "model_name": "LogisticRegression",
    "model_version": "product_v1",
    "threshold": float(product_threshold),
    "roc_auc": 0.638071,
    "precision": 0.506,
    "recall": 0.785,
    "f1": 0.615,
    "training_rows": int(len(X)),
    "feature_count": int(len(X.columns))
}

with open(artifact_dir / "product_metrics_v1.json", "w") as file:
    json.dump(product_metrics, file, indent=2)

In [15]:
product_schema = {
    "features": list(X.columns),
    "categorical_features": categorical_columns,
    "numeric_features": numeric_columns,
    "threshold": float(product_threshold)
}

with open(
    artifact_dir / "product_feature_schema_v1.json",
    "w"
) as file:
    json.dump(product_schema, file, indent=2)

print("Product model artifact paketi tamamlandı.")

Product model artifact paketi tamamlandı.
